# <span style="color:darkblue"> Assignment 2 </span>

<font size = "5">

Use the datasets to answer the following questions:


- a) Which three constructors had the highest number of total points between 1981 and 2020? How many total points did each of them get? How do the total number of points for each constructor compare to the average across constructors?

- b) Which three constructors had the highest number of total points between 2001 and 2020? How many total points did each of them get? How do the total number of points for each constructor compare to the average across constructors?

- c) How did the rankings change across the two time periods?

- d) How many different drivers did Ferrari have between 1981 and 2020?

- e) What was the best year for Ferrari between 1981 and 2020?

# <span style="color:darkblue"> Clean Data and Merge Datasets </span>

<font size = "5">

From the codebook, I identified the variables required to address the research questions. In step 1, I will load and clean the raw data, check missing value, and convert necessary data type.

Then I will extract key variables from multiple datasets and integrate them into a new dataset for further comparative analysis.

In [125]:
# library
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# read data
results = pd.read_csv("data_raw/results.csv")
races = pd.read_csv("data_raw/races.csv")
constructors = pd.read_csv("data_raw/constructors.csv")
standings = pd.read_csv("data_raw/constructor_standings.csv")

# check the types of variables in results.csv
results.dtypes

resultId             int64
raceId               int64
driverId             int64
constructorId        int64
number              object
grid                 int64
position            object
positionText        object
positionOrder        int64
points             float64
laps                 int64
time                object
milliseconds        object
fastestLap          object
rank                object
fastestLapTime      object
fastestLapSpeed     object
statusId             int64
dtype: object

<font size = "4">
In the dataset results.csv. I need "raceId", "driverId", "constructorId", "points" and "rank". The types table suggest that rank is now stored as object type, so we will it to numeric for further comparison. To achieve. this, I will identify what non-numeric values exist in the rank column, replace them and recode rank.

In [ ]:
# Based on the types of variables, I need to convert variable "rank" to numeric.
# Check what unique value in column "rank" is not numeric.
rank_subset = results.query("rank.str.isnumeric() == False")
list_unique = pd.unique(rank_subset["rank"])
print(list_unique)

['\\N']


In [ ]:
# I will substitute "\\N" with np.nan, and convert "rank" to numeric variable
results["rank"] = results["rank"].replace("\\N", np.nan)
results["rank"] = pd.to_numeric(results["rank"])

# Store the clean dataset
results.to_csv("data_clean/results.csv")

<font size = "4">
From races.csv, I will extract year and raceId. The year variable will help us better analyze constructor and driver performance across different time periods, while raceId serves as the key for merging datasets. In the following examination, both variables are already stored as integer types and requires no further cleaning.

In [ ]:
# check the types of variables in races.csv
races.dtypes

raceId          int64
year            int64
round           int64
circuitId       int64
name           object
date           object
time           object
url            object
fp1_date       object
fp1_time       object
fp2_date       object
fp2_time       object
fp3_date       object
fp3_time       object
quali_date     object
quali_time     object
sprint_date    object
sprint_time    object
dtype: object

<font size = 4>
From constructorsRef, I will extract constructorRef and constructorId. The variable constructorRef provides constructors' names, and the constructorsId is the key for merging datasets. 

In [ ]:
# check the types of variables in races.csv and constructors.csv
constructors.dtypes

constructorId      int64
constructorRef    object
name              object
nationality       object
url               object
dtype: object

<font size = "4">
I will first merge the results dataset with races, then incorporate constructor information. The dataset results_merge now contains information of constructor names and race time, and it's ready to address the analytical questions below.

In [41]:
results_races_merge = pd.merge(results[["constructorId", "driverId", "raceId", "points", "rank"]],
                         races[["raceId", "year"]],
                         on = "raceId",
                         how = "left")
results_races_merge

results_merge = pd.merge(results_races_merge,
                         constructors[["constructorId", "constructorRef"]],
                         on = "constructorId",
                         how = "left")

# <span style="color:darkred"> Question A </span>

<font size = "5">

- Which three constructors had the highest number of total points between 1981 and 2020? 
- How many total points did each of them get? 
- How do the total number of points for each constructor compare to the average across constructors? 

# <span style="color:green"> Approach </span>
<font size = "4">
To answer the questions,  I will subset the dataset to include only races occurring between 1981 and 2020, then calculate the total points accumulated by each constructor during this period. The constructors will be ranked by total points to identify the top three performers. Additionally, I will compute the mean points across all constructors to see how each constructor's performance deviates from the average.

In [148]:
# subset data between 1981 and 2020 with query.
subset_81_20 = results_merge.query("year >= 1981 and year <= 2020")
subset_81_20

,constructorId,driverId,raceId,points,rank,year,constructorRef
0,1,1,18,10.0,2,2008,mclaren
1,2,2,18,8.0,3,2008,bmw_sauber
2,3,3,18,6.0,5,2008,williams
3,4,4,18,5.0,7,2008,renault
4,1,5,18,4.0,1,2008,mclaren
...,...,...,...,...,...,...,...
24955,51,841,1047,0.0,7,2020,alfa
24956,3,849,1047,0.0,16,2020,williams
24957,210,825,1047,0.0,13,2020,haas
24958,210,850,1047,0.0,8,2020,haas


In [149]:
# group the data by constructors, calculate total points for each constructor
# sort the results in descending order by total points.
subset_81_20_agg = (subset_81_20.groupby("constructorRef")
                       .agg(sum_points = ("points", "sum"))
                        .sort_values(by = "sum_points", ascending = False))

# display the top three constructors
subset_81_20_agg.head(3)

,sum_points
constructorRef,
ferrari,7374.0
mercedes,5685.0
mclaren,5229.5


In [ ]:
# calculate the standard deviation assess performance variability across constructors
points_std = subset_81_20_agg["sum_points"].std()
points_std

1448.5024982933878

In [ ]:
# calcultate the mean of total points across constructors 
overall_mean_points = subset_81_20_agg["sum_points"].mean()

subset_81_20_agg["mean_across_constructors"] = overall_mean_points

# compute for each constructor to see how it performed compare to the average
subset_81_20_agg["difference_from_mean"] = subset_81_20_agg["sum_points"] - overall_mean_points
subset_81_20_agg

,sum_points,mean_points,mean_across_constructors,difference_from_mean
constructorRef,,,,
ferrari,7374.0,5.347353,532.238806,6841.761194
mercedes,5685.0,13.220930,532.238806,5152.761194
mclaren,5229.5,3.784009,532.238806,4697.261194
red_bull,5043.5,8.295230,532.238806,4511.261194
williams,3355.0,2.424133,532.238806,2822.761194
...,...,...,...,...
spirit,0.0,0.000000,532.238806,-532.238806
simtek,0.0,0.000000,532.238806,-532.238806
mf1,0.0,0.000000,532.238806,-532.238806


# <span style="color:purple"> Answer A</span>
<font size = "4">

Based on the analysis of data between 1981 and 2020, Ferrari, Mercedes, and McLaren are the top three constructors with the highest total points:


Ferrari: 7,374

Mercedes: 5,685

McLaren: 5,229.5


The mean total points across all constructors is 532.24 with a standard deviation of 1,448.5, which indicates significant performance disparity across teams. All three top constructors performed substantially above the average level, with Ferrari exceeding the mean by 6,842 points, Mercedes by 5,153 points, and McLaren by 4,697 points. 

# <span style="color:darkred"> Question B </span>

<font size = "5">

- Which three constructors had the highest number of total points between 2001 and 2020? 
- How many total points did each of them get? 
- How do the total number of points for each constructor compare to the average across constructors? 

# <span style="color:green"> Approach </span>

<font size = "4">
Question B follows a similar approach to Question A, but focuses on period between 2001 and 2020. To achieve this, I will filter the dataset to include only races occurring between 2001 and 2020. I will repeat the same analytical process: calculate total points accumulated by each constructor, rank them by total points, and compute the mean and standard deviation across all constructors.

In [152]:
# Subset data between 2001 and 2020 with query.
subset_01_20 = results_merge.query("year >= 2001 and year <= 2020")
subset_01_20

# Group the data by constructors, calculate total points for each constructor
# Sort the results in descending order by total points.
subset_01_20_agg = (subset_01_20.groupby("constructorRef")
                       .agg(sum_points = ("points", "sum"),
                            mean_points = ("points", "mean"))
                        .sort_values(by = "sum_points", ascending = False))

# Display the top three constructors
subset_01_20_agg.head(3)

,sum_points,mean_points
constructorRef,,
ferrari,5862.0,7.879032
mercedes,5685.0,13.220930
red_bull,5043.5,8.295230


In [154]:
# Calculate the standard deviation assess performance variability across constructors
points_std01 = subset_01_20_agg["sum_points"].std()
points_std01

1612.029956505855

In [145]:
# Calcultate the mean of total points across constructors 
overall_mean_points01 = subset_01_20_agg["sum_points"].mean()
overall_mean_points01

# Compute for each constructor to see how it performed compare to the average
subset_01_20_agg["mean_across_constructors"] = overall_mean_points01

subset_01_20_agg["difference_from_mean"] = subset_01_20_agg["sum_points"] - overall_mean_points01
subset_01_20_agg


,sum_points,mean_points,mean_across_constructors,difference_from_mean
constructorRef,,,,
ferrari,5862.0,7.879032,786.014286,5075.985714
mercedes,5685.0,13.220930,786.014286,4898.985714
red_bull,5043.5,8.295230,786.014286,4257.485714
mclaren,3284.0,4.413978,786.014286,2497.985714
williams,1535.5,2.063844,786.014286,749.485714
renault,1465.0,2.634892,786.014286,678.985714
force_india,1098.0,2.589623,786.014286,311.985714
lotus_f1,706.0,4.584416,786.014286,-80.014286
toro_rosso,500.0,0.932836,786.014286,-286.014286


# <span style="color:purple"> Answer B</span>
<font size = "4">

Based on the analysis of data between 2001 and 2020, Ferrari, Mercedes, and Red bull are the top three constructors with the highest total points:


Ferrari: 5,862

Mercedes: 5,685

Red Bull: 5,043.5


The mean total points across all constructors is 786 with a standard deviation of 1,612, which indicates even higher performance disparity across teams compared to the longer 1981-2020 period. The result also shows that the competition among top constructors has intensified and the gap between top team and the industry average has widened.

# <span style="color:darkred"> Question C </span>

<font size = "5">

How did the rankings change across the two time periods?


# <span style="color:yellow"> Approach 1 Rank</span>

<font size = "4">

The rank in the raw dataset only refers to the drivers' fastest lap rankings in each race, which allows us to track how each driver's performance changed throughout the period.

In [ ]:
# Extract key variables rank, year and driverId and create a new rank dataset
# I create a bouble index with driverId as the primary level and year as the secondary level
# We can use the dataset to track each drivers' performance across the time.

rank = (results_merge[["driverId", "year", "rank"]]
        .set_index(["driverId", "year"])
        .sort_index())
rank

rank
driverId year     
1        2007    3
         2007    1
         2007    2
         2007    2
         2007    2
...            ...
855      2022   15
         2022    7
         2022   17
         2022    8
856      2022   13

[25840 rows x 1 columns]

# <span style="color:green"> Approach 2 Constructor Rank</span>

<font size = "4">

To calculate the ranking changes, I need to merge the results subset from both time periods, 1981-2020 and 2001-2020. Since the rank variable in the raw dataset only contains driver ranking, I will generate new rankings based on total points accumulated by each constructor and analyze how these rankings change across the two periods.

In [ ]:
# Rename the sum_points in the two aggregate dataset 
# because both have the same variable name "sum_points"
subset_81_20_agg = subset_81_20_agg.rename(columns = {"sum_points": "sum_1981_2020"})
subset_01_20_agg = subset_01_20_agg.rename(columns = {"sum_points": "sum_2001_2020"})


In [170]:
# Merge the aggregated datasets from both time periods
# because the constructRef becomes index in the two dataset
# I use "left_index=True, right_index=True" parameters to join
agg_merge = pd.merge(subset_81_20_agg[["sum_1981_2020"]],
                     subset_01_20_agg[["sum_2001_2020"]],
                     left_index = True,
                     right_index = True,
                     how = "left")

# Generate ranking for both time periods 
agg_merge["rank_1981_2020"] = agg_merge["sum_1981_2020"].rank(ascending=False, method='min')
agg_merge["rank_2001_2020"] = agg_merge["sum_2001_2020"].rank(ascending=False, method='min')
agg_merge

,sum_1981_2020,sum_2001_2020,rank_1981_2020,rank_2001_2020
constructorRef,,,,
ferrari,7374.0,5862.0,1.0,1.0
mercedes,5685.0,5685.0,2.0,2.0
mclaren,5229.5,3284.0,3.0,4.0
red_bull,5043.5,5043.5,4.0,3.0
williams,3355.0,1535.5,5.0,5.0
...,...,...,...,...
spirit,0.0,NaN,51.0,NaN
simtek,0.0,NaN,51.0,NaN
mf1,0.0,0.0,51.0,30.0


# <span style="color:purple"> Answer C</span>

<font size = "4">

Based on the ranking analysis of constructors across the two time periods, the top-tier constructors showed stability in the rankings. Ferrari maintained its winner position across both periods, and Mercerdes holds the position of the second. Mclaren and Red_bull exchanged their positions of 3 and 4 as Red Bull rose from 4th to 3rd. In the top 10 constructors, only Benetton dropped from the 8th to 22nd position in the 2001-2020 period. Beyond the top 10, constructor rankings shows considerable volatility, with many teams dropped out after 2001. 

# <span style="color:darkred"> Question D </span>

<font size = "5">
How many different drivers did Ferrari have between 1981 and 2020?


# <span style="color:green"> Approach </span>

<font size = "4">
Filter data to only include the Ferrari's information, Use len and pd.unique to count how many different drivers they have between 1981 and 2020. Then, group the data by year to observe how many drivers they have in each year.

In [ ]:
# Extract ferrari's data and count unique drivers by driverId
subset_ferrari = subset_81_20.query("constructorRef == 'ferrari'")
len(pd.unique(subset_ferrari["driverId"]))
 

25

In [ ]:
# Group ferrari's data by year to count drivers in each year
grouped_ferrari = subset_ferrari.groupby("year")["driverId"].nunique()
grouped_ferrari

year
1981    2
1982    4
1983    2
1984    2
1985    3
1986    2
1987    2
1988    2
1989    2
1990    2
1991    3
1992    3
1993    2
1994    3
1995    2
1996    2
1997    2
1998    2
1999    3
2000    2
2001    2
2002    2
2003    2
2004    2
2005    2
2006    2
2007    2
2008    2
2009    4
2010    2
2011    2
2012    2
2013    2
2014    2
2015    2
2016    2
2017    2
2018    2
2019    2
2020    2
Name: driverId, dtype: int64

# <span style="color:purple"> Answer D </span>

<font size = "4">

Between 1981 and 2020, Ferrari has 25 different drivers in total, maintaining a size of 2 to 4 drivers every year.

# <span style="color:darkred"> Question E </span>

<font size = "5">
What was the best year for Ferrari between 1981 and 2020?

# <span style="color:green"> Approach 1 </span>

<font size = "4">
To decide which was the best year for Ferrari between 1981 and 2020. I will first define the criteria. In approach 1, I will calculate ferrari's total points in each year, and the year which Ferrari gets the most total points is its best year.

In [ ]:
subset_ferrari

# Aggregate ferrari's total points by year and rank by annual points
ferrari_agg = (subset_ferrari.groupby("year")
                             .agg(sum_points = ("points", "sum"),
                                  mean_points = ("points", "mean"))
                              .sort_values(by = "sum_points", ascending = False))

# Display Ferrari's anual total points
ferrari_agg

,sum_points,mean_points
year,,
2018,571.0,13.595238
2017,522.0,13.050000
2019,504.0,12.000000
2015,428.0,11.263158
2012,400.0,10.000000
2016,398.0,9.476190
2010,396.0,10.421053
2011,375.0,9.868421
2013,354.0,9.315789


# <span style="color:green"> Approach 2 </span>

<font size = "4">
In approach 2, I will define Ferrari's best year as the season with the most wins. I will extract the wins variable from constructor_standings.csv, merge it with raceId and year from races.csv, and with constructorId and constructorRef from constructors.csv to create a new dataset. After filtering for Ferrari's data, I will aggregate wins by year to identify the year with Ferrari's highest wins count.

In [ ]:
# Check the data type of wins variable
standings.dtypes

constructorStandingsId      int64
raceId                      int64
constructorId               int64
points                    float64
position                    int64
positionText               object
wins                        int64
dtype: object

In [ ]:
# Merge the dataset standings and races to include time information
wins_merge = pd.merge(standings,
                      races[["raceId", "year"]],
                      on = "raceId",
                      how = "left")

In [ ]:
# Create a new dataset by merging standings, races and constructors
wins = pd.merge(wins_merge,
                constructors[["constructorId", "constructorRef"]],
                on = "constructorId",
                how = "left")

# Filter for Ferrari
wins_ferrari = wins.query("constructorRef == 'ferrari'")

# Sort data by wins counts
wins_ferrari.sort_values(by = "wins", ascending = False)

,constructorStandingsId,raceId,constructorId,points,position,positionText,wins,year,constructorRef
898,6297,106,6,254.0,1,1,15,2004,ferrari
908,6307,107,6,262.0,1,1,15,2004,ferrari
1246,5971,140,6,221.0,1,1,15,2002,ferrari
1235,5960,139,6,205.0,1,1,14,2002,ferrari
888,6287,105,6,244.0,1,1,14,2004,ferrari
...,...,...,...,...,...,...,...,...,...
7758,22239,592,6,12.0,2,2,0,1974,ferrari
7746,22228,591,6,12.0,2,2,0,1974,ferrari
7735,22217,590,6,6.0,2,2,0,1974,ferrari
7572,22503,579,6,8.0,4,4,0,1975,ferrari


<font size = "4">
The wins_ferrari dataset above shows multiple wins values for each year because the data contains individual race records. Therefore, I need to aggregate the wins by year.

In [ ]:
# Group Ferrari's data by year and calculates the total wins for each year
# Sort the aggreegated data in descending order by total wins
wins_agg = (wins_ferrari.groupby("year")
                        .agg(sum_wins = ("wins", "sum"))
                        .sort_values(by = "sum_wins", ascending = False))
wins_agg

,sum_wins
year,
2004,152
2002,126
2000,87
2008,87
2001,84
...,...
1969,0
1967,0
1965,0


# <span style="color:purple"> Answer E </span>

<font size = "4">

To determine Ferrari's best year between 1981 and 2020, I analyzed their performance using two different criteria: total points and total wins. 


And the two standards yield different results. For overall performance and points accumulation, 2018 is Ferrari's best year that it obtains 571 points. But for race domination and wins counts, Ferrari's best year is 2004 with 152 total wins.
